# Resampling and CV - How to Compare Models Without Fooling Yourself

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb08_cross_validation_model_comparison_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Write the $k$-fold cross-validation estimator for regression (MSE) and classification (misclassification), and plug any score — $R^2$, ROC-AUC — into the same average
2. Run 5-fold CV on a regression business case (California Housing) and on a classification business case (Breast Cancer), reporting the five fold scores, their mean, and their standard deviation every time
3. Picture the fold scores and set a single validation score against the CV mean plus or minus the margin you declared — as a description of one split, never as a test
4. Compare a challenger with the current champion on identical folds using `compare_paired`: per-fold gains, an interval corrected for fold overlap, and one of **four outcomes** read against a margin $\delta$ declared before any score is computed
5. Declare a margin band (small / base / large) in the stakeholder's terms, write the four-point **champion statement**, and set up the one-split, five-fold pattern every notebook from nb09 on uses

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: One Lucky Split or Genuinely Better?

Two stakeholders are waiting on your answer this week, one from each half of the course so far. Both are asking the same question in different language.

**HomeValue Analytics (California Housing — regression).** Your Ridge pipeline from nb05 posts a validation $R^2$ of about **0.62** on one split. The CFO is not rude about it, but she is skeptical: *"If we had drawn a different 20% for the validation set, would the number have come out the same?"* With only one split in hand, the honest answer is *you cannot tell*.

**MedScreen (Breast Cancer — classification).** Your logistic regression from nb07 posts a validation ROC-AUC above **0.99**. The chief medical officer is similarly polite and similarly unconvinced: *"Is that 0.99 real, or did we just get lucky with the 114 patients in the validation set?"* Same shape of question, same problem: one split cannot answer it.

**Same question in two domains, same solution — cross-validation.** The trick is to stop relying on *one* 60/20/20 split. Repartition the training data several different ways, train and score once per partition, and study the **five scores** instead of a single number. The story you report back stops being *"the model got 0.62"* and becomes *"across five different held-out slices the model averaged 0.59, and the slices ranged from 0.55 to 0.61"* — a sentence a decision-maker can act on.

Then comes the question both stakeholders really care about. Somebody always proposes a different model, and the CFO does not ask *"how good is it?"* — she asks *"is it better than what we run today by enough to matter?"* That is a question about the **gain** of one model over another, fold by fold, judged against a margin the stakeholder sets before anyone looks at the scores.

> **Today's focus:** run $k$-fold CV on both business cases and report the fold scores with their mean and spread; set the single validation score against the CV mean; then compare a challenger with the current model on identical folds with `compare_paired`, read one of four outcomes against a margin you declared in advance, and write the result as a champion statement.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# The course's one model-comparison tool, used unchanged in every notebook from here on.
# A = the challenger, B = the current champion. Both score arrays must come from ONE splitter object.
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# Decision margins, declared BEFORE any model is scored. The base margin decides the champion;
# small and large are a sensitivity check (would the answer change for a more relaxed or a more
# demanding stakeholder?). These are stated business assumptions, not facts about the data.
DELTAS_R2 = {'small': 0.005,   # HomeValue: a CFO who switches models cheaply
             'base':  0.01,    # the base scenario — decides the champion
             'large': 0.02}    # a CFO for whom every model change triggers a costly re-validation
DELTAS_AUC = {'small': 0.005,  # MedScreen: a review board that switches for any measurable ranking gain
              'base':  0.01,   # the base scenario — one more correctly ordered malignant-benign pair per 100
              'large': 0.02}   # a board that strongly values the model a clinician can explain
DELTA_R2, DELTA_AUC = DELTAS_R2['base'], DELTAS_AUC['base']

print("✓ Setup complete!")
print(f"✓ Margins declared before scoring — R²: {DELTAS_R2} | ROC-AUC: {DELTAS_AUC}")

**Reading the output:**

The setup cell gathers every tool the rest of the notebook uses, so nothing downstream has to stop and re-import. Here is a short tour of what each piece does and where you will meet it.

The **two splitters** — `KFold` and `StratifiedKFold` — are the objects that decide which rows land in which fold. You will pass one of them to `cross_val_score` and it takes care of the splitting internally. `KFold` chops the training data into $k$ equal-sized chunks at random, which is what you want in Section 2 for the California Housing regression problem. `StratifiedKFold` does the same thing but preserves the positive-class rate of the target in every chunk — which will matter the moment you move to classification in Section 3, because a fold with zero malignant patients would make ROC-AUC undefined.

The **two CV convenience functions** — `cross_val_score` and `cross_validate` — wrap the train-evaluate loop so you never have to write it by hand. `cross_val_score` returns one score per fold (a NumPy array of length $k$); `cross_validate` returns more information (train scores, fit times, multiple metrics) when you ask for it. The rest of this notebook uses `cross_val_score`, and it keeps every fold array: you will need the individual fold scores, not only their average, the moment you compare two models.

**`train_test_split`** is the same single-split tool from nb01–nb07. Today it carves the familiar 60/20/20 partition one last time, to show why a single validation score is not enough; Section 5 replaces it with the one 80/20 split every notebook from nb09 on uses.

**`compare_paired`** is the course's one tool for deciding between two models, and you will use it, unchanged, in every notebook that compares models. It takes the fold scores of a challenger (A) and of the current champion (B) — both computed on the *same* folds — subtracts them fold by fold, puts an interval around the mean gain, and reads that interval against a margin. Section 4 takes it apart line by line. **`scipy.stats`** is here for one call inside it, `stats.t.ppf(0.975, k - 1)`, the Student's $t$ value that sets the width of that interval.

The **margins** — `DELTAS_R2` and `DELTAS_AUC` — are written down before a single model is scored, and that ordering is the point. Each case gets three: the **base** margin decides, and the small and large ones check whether the decision would change for a stakeholder who is more relaxed or more demanding about switching models.

The **three models** — `LinearRegression`, `Ridge`, `LogisticRegression` — are already familiar from nb03–nb07. Nothing new about the models today; what is new is how you *evaluate and compare* them.

`RANDOM_SEED = 474` is inherited from nb01–nb07, so every split and every fold you see is directly comparable to what you computed in earlier notebooks.

> **A question that often comes up at this point:** *"Do I need to memorize which splitter goes with which problem type?"* No — just remember one rule: **if the target is categorical, stratify**. Everything else follows.

---

## 1. Why Cross-Validation Exists

Before the first line of CV code runs, take a minute to build up the intuition for *why* this technique exists. The problem is simple, the fix is elegant, and the math is just a formal rewrite of something you are about to watch in an animation. Most of the work today is reading and interpreting; the coding is the easy part.

### The problem with a single split

A single 60/20/20 split gives you one number for validation performance — but that number is **one draw from a distribution**. Which 20% of the data ended up in the validation set is a random choice, and a *different* random choice would give a *different* score. With 114 validation patients (breast cancer) or roughly 4,128 validation tracts (California Housing), the variation from one split to the next is easily large enough to flip the ordering of two candidate models. That is the CFO's and the chief medical officer's worry, stated in plain English: *"would the number change if we had drawn differently?"* — and the answer with one split is always *"yes, a bit, and we cannot tell by how much."*

### $k$-fold CV — the mechanism (watch it happen)

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/KfoldCV.gif" width="640" alt="Animated visualization of 5-fold cross-validation">
</center>

Every frame of the animation above is one iteration of $k$-fold CV. The training data is sliced into $k$ non-overlapping **folds**; at each iteration exactly one fold is held out for scoring (the colored stripe) and the other $k-1$ folds are used for fitting the model. The held-out fold rotates through all $k$ positions, so by the end of the cycle **every sample in the training set has been predicted exactly once, by a model that never saw it during fitting**. That is the whole trick — a single pass of the animation gives you $k$ honest validation scores instead of one.

Three properties are worth naming while you watch:

- **Fold size is roughly constant.** Each fold is about $n/k$ rows. With about 12,400 training tracts and $k=5$, each fold holds around 2,500 rows.
- **The test set is never touched.** Everything you see rotating belongs to the *training* portion of the 60/20/20 split. The 20% test set stays locked away until nb14 or the final project — exactly the discipline you built in nb01.
- **The $k$ scores are not independent.** Adjacent iterations share $k-2$ of their $k-1$ training folds, so their scores are positively correlated. That matters more than it looks. The familiar $\text{SD}/\sqrt{k}$ arithmetic assumes the $k$ numbers are independent draws, so when the folds share training rows it **understates** how much the estimate would move on fresh data, and switching to Student's $t$ does not repair that — $t$ only accounts for having estimated the standard deviation from $k$ numbers. That is why this course does not put a 95% interval around a single model's CV score (Section 2 explains), and why the one interval you *will* compute — around the per-fold gain of one model over another, in Section 4 — carries an extra correction term for the shared rows.

> **A question that often comes up here:** *"Why not just take a bigger validation set instead of doing all this?"* Because a single validation set — no matter how big — still comes from one random partition and gives you one number. Cross-validation gives you $k$ numbers from $k$ partitions, so you can see how much the score moves from one slice of the data to the next. That spread is what turns *"the model got 0.62"* into *"the model averaged 0.59 across five folds, and the folds ranged from 0.55 to 0.61"* — the numbers you are about to compute in Section 2.

### The $k$-fold CV estimator

With the mechanism in your head, the equations are just a formal way of writing down what you already watched. Partition the training data into $k$ equally-sized folds — call them $C_1, \ldots, C_k$. For each fold $i$, train on the other $k-1$ folds and evaluate on the held-out fold $C_i$. Average the $k$ scores.

**For regression** (ISLP eq. 5.3):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{MSE}_i
\qquad \text{where} \qquad
\text{MSE}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \left( y_j - \hat{y}_j^{(-i)} \right)^2$$

The notation $\hat{y}_j^{(-i)}$ just means *"the prediction for sample $j$ from the model trained on the $k-1$ folds that do not contain $j$."* That is the same idea the animation showed, written with subscripts.

**For classification** (ISLP eq. 5.4):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{Err}_i
\qquad \text{where} \qquad
\text{Err}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \mathbb{1}\!\left( y_j \neq \hat{y}_j^{(-i)} \right)$$

The structure is identical — only the per-fold loss changes (squared error for regression, misclassification rate for classification). Accuracy is one minus a fold's misclassification rate, and a fold's $R^2$ is one minus its MSE divided by the spread of $y$ in that fold, so both are direct rewrites of these losses. ROC-AUC and F1 are *not* rewrites of the misclassification rate — ROC-AUC measures how well a fold's predictions rank the cases, F1 balances precision and recall at one threshold — but every one of them is computed fold by fold and plugs into the exact same average:

$$\text{CV}_{(k)}^{\text{score}} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{score}_i$$

where $\text{score}_i$ is whatever metric you pass into the `scoring=` argument of `cross_val_score` (e.g., `'r2'`, `'roc_auc'`, `'f1'`).

> **Worth noticing:** these equations really are "just averages." The interesting questions are never *"how do I average $k$ numbers?"* — they are *"how do I report the spread of those $k$ numbers honestly?"* and *"how do I tell whether a second model is better by enough to matter?"* The summary statistics below answer the first; Section 4 answers the second.

### The limiting case — Leave-One-Out CV (LOOCV)

One natural question at this point: *"what happens if I push $k$ all the way up to $n$?"* That limiting case has a name and a good visualization.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/LOOCV.gif" width="640" alt="Animated visualization of Leave-One-Out cross-validation">
</center>

When $k = n$, every fold is a single row. That special case is **Leave-One-Out CV (LOOCV)**: train on $n-1$ samples, predict the one that was left out, repeat $n$ times. The animation shows every row taking a turn as the held-out sample.

Two properties fall out of the $k = n$ setting, and together they explain why LOOCV is *not* the default in this course:

- **Low bias, because each training fold uses nearly all the data.** The model fit on $n-1$ rows is almost the model you would deploy. This makes LOOCV (nearly) unbiased as an estimator of the true test error.
- **High variance of the estimate, and very high compute cost.** The $n$ training sets differ by only one row, so the $n$ held-out errors are highly correlated, which inflates the *variance of the LOOCV mean*. And training $n$ models — about 12,400 for California Housing — is several orders of magnitude slower than training 5 or 10.

> **A common follow-up question:** *"If LOOCV is more accurate, why doesn't everyone use it?"* Because "more accurate" on paper loses to "much faster and almost as good" in practice. A small $k$ trades a little more bias for visibly lower variance in the mean, and for compute that finishes while you are still thinking about the result. This course uses $k = 5$ **everywhere** — for every model, every grid search, and every comparison — so that every interval you compute is built the same way and every comparison in one notebook can be set beside a comparison in another.

### Summary statistics — always report the fold scores, their mean, and their spread

After every CV run you end up with $k$ fold scores $s_1, \ldots, s_k$. Report the scores themselves (five numbers are easy to show) and two summaries of them, *every single time*.

**Mean** — the point estimate:

$$\bar{s} \;=\; \frac{1}{k} \sum_{i=1}^{k} s_i$$

**Standard deviation** across folds — how much the score wiggles from one partition to the next. Use the sample SD with $k-1$ in the denominator:

$$\text{SD} \;=\; \sqrt{\frac{1}{k-1} \sum_{i=1}^{k} \bigl(s_i - \bar{s}\bigr)^2}$$

Notice what is *not* on the list: a 95% interval around the mean. The fold scores are not independent, so the textbook $\bar{s} \pm t \cdot \text{SD}/\sqrt{k}$ comes out too narrow, and — more importantly — the decisions in this course are never about one model in isolation. The one interval you will compute is the one around the per-fold **gain** of a challenger over the current model, corrected for the rows the folds share (Section 4).

> **The rule the rest of this notebook follows, no exceptions:** always report the fold scores, their mean, and their SD together, with a picture of the fold scores so the spread is visible. **Never quote a CV mean without its spread.**

With the mechanism, the equations, and the summary statistics in hand, you are ready to apply the recipe to real data. Section 2 takes the regression case (California Housing); Section 3 does the same thing for classification (Breast Cancer); Section 4 turns the fold scores of two models into a decision.

---

## 2. $k$-Fold CV for Regression — California Housing (HomeValue Analytics)

Enough theory. Time to watch it in action.

First stop: the HomeValue Analytics pricing model from nb01–nb05. You already have a Ridge pipeline (StandardScaler + Ridge, $\alpha = 1.0$) that posts a validation $R^2$ of about **0.62** on one 60/20/20 split. Today the CFO wants to know how much that number moves when the split changes.

### Declare the margins before you score anything

Before the first fold is scored, write the CFO's three margins down — they are already in the setup cell as `DELTAS_R2`. **Base: 0.01 $R^2$** decides every HomeValue comparison. **Small: 0.005 $R^2$**, for a CFO who switches models cheaply. **Large: 0.02 $R^2$**, for a CFO for whom every model change triggers a costly re-validation. These are assumptions about how the CFO decides, not facts about the data, and you will use the same three numbers, unchanged, for every HomeValue comparison through nb14. The cell below also translates each one into dollars: how much a gain of that size would shorten the typical miss (the RMSE) per tract at today's level of fit. That translation describes the metric; it is not money saved.

### The four-step recipe

The plan is a four-step recipe that will repeat, step for step, on the classification problem in Section 3. If you can walk through these four steps here, you already know Section 3 — only the splitter and the scoring metric will change:

1. **Split the data.** Apply the usual 60/20/20 partition with `RANDOM_SEED = 474` (same split nb01–nb05 used, so the numbers are directly comparable). This three-way split appears today only to show why one validation score is noisy.
2. **Score the single split.** Fit Ridge on the training set, score once on the validation set — and record that **single-split $R^2$**. This is the "one draw from a distribution" you are about to compare against a full CV run.
3. **Run 5-fold CV on the training set only.** The test set stays locked. Report the five fold scores, their mean, and their standard deviation (`ddof=1`).
4. **Picture and describe.** Plot the five fold scores on the left, and the single-split score beside the CV mean on the right. A one-line status print says whether the single-split $R^2$ landed within the base margin of the CV mean (INSIDE), above it, or below it — shorthand for how close that one split came to the fold scores, not a test of anything.

Plugging $R^2$ into the CV estimator from Section 1 gives the equation this section is about to evaluate numerically:

$$\text{CV}_{(5)}^{R^2} \;=\; \frac{1}{5} \sum_{i=1}^{5} R^2_i$$

where $R^2_i$ is the coefficient of determination on held-out fold $i$. Five numbers, their average, and their spread. That is it.

> 💡 **Gemini Prompt:** "Load the California Housing dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a Ridge pipeline (StandardScaler + Ridge with alpha=1.0), fit it on `X_train`, and compute the single-split validation R² on `X_val`. Then run 5-fold `KFold` cross-validation on the training set with `scoring='r2'` and keep the five fold scores. Print the fold scores, their mean, and their sample standard deviation (`ddof=1`). Using the variance of the training target `v`, translate each margin in `DELTAS_R2` into dollars of typical error at the CV mean with `1e5 * (sqrt(v*(1-R2)) - sqrt(v*(1-R2-delta)))`. Render a 1x2 matplotlib figure: left panel, the five fold scores as dots with a dashed line at their mean; right panel, two bars — the single validation R² and the 5-fold CV mean — with dotted lines at the CV mean plus and minus `DELTA_R2`. End with one printed line saying whether the single validation R² is within `DELTA_R2` of the CV mean (INSIDE), above that band, or below it."
>
> **After running, verify:**
> - Single-split validation R² is printed as one number
> - Five fold R² values are printed, with their mean and SD at 4-decimal precision
> - No interval is computed or drawn around the mean — the spread is reported as the SD
> - The three margins are printed with their dollar meaning
> - Left panel: 5 dots and a red dashed mean line; right panel: 2 bars and two dotted lines at the CV mean ± δ
> - A final status line prints INSIDE, ABOVE, or BELOW
> - Test set remains untouched
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# --- 1. Load California Housing and do the same 60/20/20 split used across nb01-nb05 ---
# (The three-way split appears here only to show why one validation score is noisy;
#  Section 5 replaces it with the one 80/20 split every notebook from nb09 on uses.)
california = fetch_california_housing(as_frame=True)
X_reg = california.data
y_reg = california.target

X_reg_temp, X_reg_test, y_reg_temp, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
print(f"Train: {len(X_reg_train)} | Val: {len(X_reg_val)} | Test: {len(X_reg_test)} (locked)")

# --- 2. Fit Ridge on train, score once on the single validation split ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0, random_state=RANDOM_SEED))
])
ridge_pipeline.fit(X_reg_train, y_reg_train)
val_r2 = ridge_pipeline.score(X_reg_val, y_reg_val)
print(f"\nSingle-split validation R²: {val_r2:.4f}")

# --- 3. Run 5-fold CV on the TRAINING portion only (test set stays locked) ---
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)   # keep this object: every comparison reuses it
cv_scores = cross_val_score(ridge_pipeline, X_reg_train, y_reg_train, cv=cv, scoring='r2')

mean_r2 = cv_scores.mean()
sd_r2 = cv_scores.std(ddof=1)

print(f"\n=== 5-FOLD CV (Ridge on California Housing) ===")
print(f"Fold R²:   {np.round(cv_scores, 4)}")
print(f"Mean R²:   {mean_r2:.4f}")
print(f"SD (k-1):  {sd_r2:.4f}   (the fold-to-fold spread)")

# --- What the declared margins mean at this level of fit ---
# A gain of delta in R² shortens the typical miss (RMSE) by 1e5 * [sqrt(v(1-R²)) - sqrt(v(1-R²-delta))],
# with v = variance of the target (in units of $100,000). It translates the metric; it is not money saved.
v = y_reg_train.var()
print(f"\nDeclared margins, translated at a CV R² of {mean_r2:.4f}:")
for name, delta in DELTAS_R2.items():
    dollars = 1e5 * (np.sqrt(v * (1 - mean_r2)) - np.sqrt(v * (1 - mean_r2 - delta)))
    print(f"  {name:>5}: δ = {delta:.3f} R²  ≈ ${dollars:,.0f} less typical error per tract")

# --- 4. Picture of the fold scores + single split vs CV mean ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the five fold scores themselves, with their mean
folds = np.arange(1, k + 1)
axes[0].scatter(folds, cv_scores, s=90, color='steelblue', edgecolor='black', zorder=3)
axes[0].axhline(mean_r2, color='red', linestyle='--', label=f'Mean = {mean_r2:.4f}  (SD = {sd_r2:.4f})')
axes[0].set_xticks(folds)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('R² score')
axes[0].set_title('Five fold scores — Cal Housing, Ridge', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower left')

# Right: the single validation split beside the CV mean, with the declared margin around the mean
labels = ['Single validation split', '5-fold CV (mean)']
values = [val_r2, mean_r2]
axes[1].bar(labels, values, color=['goldenrod', 'steelblue'], edgecolor='black')
axes[1].axhline(mean_r2 + DELTA_R2, color='gray', linestyle=':', label=f'CV mean ± δ (δ = {DELTA_R2}, the declared margin)')
axes[1].axhline(mean_r2 - DELTA_R2, color='gray', linestyle=':')
axes[1].set_ylabel('R² score')
axes[1].set_title('Single Validation R² vs. 5-fold CV Mean', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right')
for i, val in enumerate(values):
    axes[1].text(i, val + 0.005, f'{val:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Status line for the CFO — a description of how close one split landed to the fold scores, not a test
gap = val_r2 - mean_r2
status = 'INSIDE' if abs(gap) <= DELTA_R2 else ('ABOVE' if gap > 0 else 'BELOW')
print(f"\n→ Single validation R² ({val_r2:.4f}) sits {gap:+.4f} from the CV mean: {status} "
      f"the band CV mean ± δ = [{mean_r2 - DELTA_R2:.4f}, {mean_r2 + DELTA_R2:.4f}].")

**Reading the output:**

You now have, side by side, everything the CFO would want to see.

The **five dots on the left** are Ridge's $R^2$ on five different held-out fifths of the training tracts, each scored by a model fit on the other four fifths. They run from **0.5470** (fold 5) to **0.6122** (fold 3), with a mean of **0.5863** and a standard deviation of **0.0259**. Fold 5 is a noticeably harder slice of California than the other four, and that is exactly the kind of variation a single split hides: depending on which fifth you happened to hold out, you would have told the CFO anything from 0.55 to 0.61. The SD is the number to quote beside the mean, because it says how much the score moves from one slice of tracts to the next.

The **right panel** sets the single split against the CV mean. The single validation $R^2$ of **0.6169** sits **+0.0305** above the CV mean — three times the CFO's base margin, and above every one of the five fold scores — so the status line prints **ABOVE**. Read it as a description: the 4,128 validation tracts this split happened to draw were easier than a typical fifth of the data, and a report built on that one number would have promised the CFO more than the model delivers on average. The three possible statuses all describe one split, never whether the model will hold up on new tracts:

- **INSIDE** — the single split landed within $\pm\delta$ of the CV mean; either number tells the same story.
- **ABOVE** — the single split came out *high*, as it did here; quote the CV mean.
- **BELOW** — the single split came out *low*; again quote the CV mean with its spread.

The printed margins translate the CFO's band into the language of her budget: at a CV $R^2$ of 0.5863, a gain of 0.005 / **0.01** / 0.02 $R^2$ would shorten the typical miss by about **\$448 / \$899 / \$1,810** per tract. At the higher levels of fit you will reach with forests in nb12, the same $R^2$ margins are worth more dollars, which is one reason the margin is declared in $R^2$ and translated each time.

> **A question that often comes up here:** *"Why don't we put a 95% confidence interval around the model's CV score, the way a statistics course would?"* Because the arithmetic `mean ± t·s/√k` assumes the five fold scores are independent, and they are not: any two folds share three-quarters of their training rows, so the scores move together and the interval comes out too narrow — Bengio and Grandvalet (2004) showed there is no unbiased way to estimate that variability from one run of k-fold CV, and Bates, Hastie and Tibshirani (2024) found such intervals cover the truth far less often than 95%. Worse, the question the HomeValue CFO actually asks is never "how good is Ridge?" in isolation but "is Ridge better than the OLS model we run today by enough to matter?", and two separate intervals cannot answer that: they can overlap generously while one model wins on every fold. So you report each model's fold scores, mean and spread, and you put the interval where it belongs — around the per-fold *gain* of one model over the other, corrected for the shared training rows, read against the margin you declared.

A few interpretation points worth carrying forward:

- **The CV mean with its SD is what belongs on the CFO's slide.** The single validation $R^2$ is one draw; the CV mean is an average of five, and the SD shows how far apart those five were.
- **The SD describes; it does not certify.** A small SD says the five slices agreed; a large one says the estimate depends heavily on which rows you held out. Neither is a guarantee about new data.
- **The test set is still locked.** Both bars come from the *training* population; the 4,128 test tracts are reserved for the one-shot check of the final champion in nb14.

**Key takeaway:** Section 2 executed the four-step recipe for the regression side. Section 3 now does the exact same four steps on Breast Cancer — only the splitter (`StratifiedKFold`) and the scoring metric (`'roc_auc'`) will change.

---

## 3. Stratified $k$-Fold CV for Classification — Breast Cancer (MedScreen)

Same recipe, different problem. Section 2 put Ridge's CV mean at 0.5863 with a fold SD of 0.0259 — and showed one validation split running high; now apply the identical four steps to the MedScreen pipeline from nb06–nb07. If the steps feel familiar, that is the point — the CV ritual is the *same* whether you are predicting home values or classifying tumors.

There is exactly **one new ingredient**: when the target is categorical, the folds need to *stratify*. Here is why, in plain language.

The breast cancer training set has 341 patients, about 63% benign and 37% malignant. A plain random 5-fold split could — just by luck — produce one fold with only 30% malignant patients and another with 45%. That kind of fold-to-fold imbalance is exactly the *noise* cross-validation is supposed to average out, not add. `StratifiedKFold` fixes this by enforcing the 37/63 ratio in every fold, so each fold faces a representative slice of the cancer-vs-healthy mix the classifier will see in production.

> **A question that often comes up here:** *"Why didn't we need stratify for the regression case?"* Because in regression there is no "class balance" to preserve — the target is a continuous number, not a category. With classification, the minority class could in principle vanish from a fold entirely (making a metric like ROC-AUC undefined), and stratify is the simple, foolproof defense.

Following nb07, the scoring metric is **ROC-AUC** — the threshold-independent headline for ranking biopsies; the operating threshold stays a separate, cost-based decision (nb07). Before scoring, write the review board's three margins down (they are `DELTAS_AUC` in the setup cell). ROC-AUC is the probability that a randomly chosen malignant biopsy is ranked riskier than a randomly chosen benign one, so the margins have a concrete meaning: **base 0.01** is one more correctly ordered malignant–benign pair in every hundred; **small 0.005** is for a board that would switch for any measurable ranking gain; **large 0.02** is for a board that strongly values a model a clinician can explain.

Plugging AUC into the $k$-fold estimator gives:

$$\text{CV}_{(5)}^{\text{AUC}} \;=\; \frac{1}{5} \sum_{i=1}^{5} \text{AUC}_i$$

where $\text{AUC}_i$ is the ROC-AUC on held-out stratified fold $i$. The four-step recipe is unchanged: single-split validation score, 5-fold CV on training data only, fold scores with their mean and SD, a picture with interpretation. One line is new: the cell also prints each fold's **headroom**, $1 - \text{AUC}_i$ — how much room that fold leaves for any other model to do better.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED` and `stratify=y` (both calls). Build a logistic regression pipeline (StandardScaler + LogisticRegression with `max_iter=1000`), fit it on `X_train`, and compute the single-split validation ROC-AUC on `X_val` using `roc_auc_score` with `predict_proba(X_val)[:, 1]`. Then run 5-fold `StratifiedKFold` CV on the training set with `scoring='roc_auc'` and keep the five fold scores. Print the fold scores, their mean, their sample SD (`ddof=1`), and the headroom `1 - score` on each fold. Render a 1x2 matplotlib figure: left panel, the five fold scores as dots with a dashed line at their mean; right panel, two bars — the single validation ROC-AUC and the CV mean — with dotted lines at the CV mean plus and minus `DELTA_AUC`. End with one printed line saying whether the single validation ROC-AUC is within `DELTA_AUC` of the CV mean (INSIDE), above that band, or below it."
>
> **After running, verify:**
> - Single-split validation ROC-AUC is printed as one number (above 0.99)
> - Five fold AUC values are printed, with their mean and SD at 4-decimal precision, plus the headroom on each fold
> - No interval is computed or drawn around the mean
> - Left panel: 5 dots and a red dashed mean line; right panel: 2 bars and two dotted lines at the CV mean ± δ
> - Stratification preserved the 37/63 class balance in each fold (no crash, no undefined AUC)
> - A final status line prints INSIDE, ABOVE, or BELOW
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# --- 1. Load breast cancer data and apply the same stratified 60/20/20 split used in nb06/nb07 ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_temp, X_clf_test, y_clf_temp, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_clf_train, X_clf_val, y_clf_train, y_clf_val = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
print(f"Train: {len(X_clf_train)} | Val: {len(X_clf_val)} | Test: {len(X_clf_test)} (locked)")

# --- 2. Fit logistic regression on train, score once on validation ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])
log_pipeline.fit(X_clf_train, y_clf_train)
val_auc = roc_auc_score(y_clf_val, log_pipeline.predict_proba(X_clf_val)[:, 1])
print(f"\nSingle-split validation ROC-AUC: {val_auc:.4f}")

# --- 3. Run 5-fold STRATIFIED CV on the training set ---
k = 5
cv_strat = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)   # keep: comparisons reuse it
cv_scores_clf = cross_val_score(log_pipeline, X_clf_train, y_clf_train,
                                cv=cv_strat, scoring='roc_auc')

mean_auc = cv_scores_clf.mean()
sd_auc = cv_scores_clf.std(ddof=1)

print(f"\n=== 5-FOLD STRATIFIED CV (LogReg on Breast Cancer) ===")
print(f"Fold AUC:  {np.round(cv_scores_clf, 4)}")
print(f"Mean AUC:  {mean_auc:.4f}")
print(f"SD (k-1):  {sd_auc:.4f}   (the fold-to-fold spread)")
print(f"Headroom:  {np.round(1 - cv_scores_clf, 4)}   (1 − AUC: room left for any other model on each fold)")

# --- 4. Picture of the fold scores + single split vs CV mean ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

folds = np.arange(1, k + 1)
axes[0].scatter(folds, cv_scores_clf, s=90, color='steelblue', edgecolor='black', zorder=3)
axes[0].axhline(mean_auc, color='red', linestyle='--', label=f'Mean = {mean_auc:.4f}  (SD = {sd_auc:.4f})')
axes[0].set_xticks(folds)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Five fold scores — Breast Cancer, LogReg', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower left')

labels = ['Single validation split', '5-fold CV (mean)']
values = [val_auc, mean_auc]
axes[1].bar(labels, values, color=['goldenrod', 'steelblue'], edgecolor='black')
axes[1].axhline(mean_auc + DELTA_AUC, color='gray', linestyle=':', label=f'CV mean ± δ (δ = {DELTA_AUC}, the declared margin)')
axes[1].axhline(mean_auc - DELTA_AUC, color='gray', linestyle=':')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Single Validation ROC-AUC vs. 5-fold CV Mean', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right')
for i, val in enumerate(values):
    axes[1].text(i, val + 0.002, f'{val:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Same description-only status as Section 2: how close one split landed to the folds
gap_clf = val_auc - mean_auc
status_clf = 'INSIDE' if abs(gap_clf) <= DELTA_AUC else ('ABOVE' if gap_clf > 0 else 'BELOW')
print(f"\n→ Single validation ROC-AUC ({val_auc:.4f}) sits {gap_clf:+.4f} from the CV mean: {status_clf} "
      f"the band CV mean ± δ = [{mean_auc - DELTA_AUC:.4f}, {mean_auc + DELTA_AUC:.4f}].")

**Reading the output:**

The **five dots on the left** are the logistic model's ROC-AUC on five held-out stratified folds of the breast cancer training data: **1.0000, 0.9916, 0.9907, 1.0000, 0.9853**, with a mean of **0.9935** and an SD of **0.0064**. Two of the five folds are perfect — on those slices the model ranks every malignant biopsy above every benign one.

The **right panel** lines the single-split validation ROC-AUC (nb07's workflow) up against the CV mean. The single split scored **0.9974**, **+0.0038** from the CV mean, well within the board's base margin of 0.01, so the status line prints **INSIDE**: here the one split told about the same story as the five folds. As in Section 2, that is a description of how close two numbers are, not evidence that the classifier will hold up on the next 114 patients.

The **headroom line** is the new piece, and it matters for every comparison MedScreen will run. The headroom on the five folds is **0.0000, 0.0084, 0.0093, 0.0000, 0.0147**: on folds 1 and 4 no model can score higher than the logistic model does, because there is nothing left to gain. Any challenger's per-fold gain on those two folds is zero at best. Section 4 shows what that does to the board's decision.

> **A question that often comes up here:** *"If two folds are already perfect, can any model ever beat this one?"* On this evidence, not by a margin the board would accept — and saying that out loud is part of the analysis. A challenger can at best tie on folds 1 and 4, so its five per-fold gains always include two values at or below zero, and an interval built from those five gains cannot sit entirely above a positive margin; Section 4 computes the best a *perfect* challenger could show. The explainable logistic model would then stay because this evidence cannot show anything better, not because it has been proven the best model possible. If the board ever wants a decision to rest on something the AUC cannot see — say, missed cancers at a fixed review workload — that has to be declared as the deciding metric *before* the next comparison, not used to overturn this one.

A few interpretation points worth carrying forward:

- **The CV mean with its SD is what belongs on the chief medical officer's slide.** The single validation AUC is one draw; the CV mean is an average of five.
- **Stratification keeps every fold representative.** Every fold preserves the 37/63 malignant/benign ratio, so no fold can run out of malignant patients and leave AUC undefined.
- **The nb07 rule still holds.** AUC is the headline for *which classifier ranks best*. Precision and recall at a chosen threshold still come from a separate, cost-based operating decision.
- **Test set stays locked.** Both bars come from training-side data; the 114 test patients are untouched, reserved for the one-shot check of the final champion in nb14.

**Key takeaway:** Sections 2 and 3 executed the same four-step ritual on regression and classification data. The only differences were the splitter (`KFold` vs `StratifiedKFold`) and the scoring metric (`'r2'` vs `'roc_auc'`). Section 4 now turns fold scores into the decision every CFO, CMO, and project board eventually asks you to make: should a challenger replace the model you run today?

---

## 4. Comparing Two Models — Paired Gains and the Four Outcomes

Every comparison in the rest of the course has the same shape. There is a **current champion** — the model the stakeholder runs today (for HomeValue, plain OLS; for MedScreen, `LogisticRegression(C=1)`) — and a **challenger** that someone proposes instead. The question is never *"which number is bigger?"* but *"is the challenger better than the champion by enough to matter?"*

**Step 1 — score both models on the identical folds and subtract fold by fold.** Pass the *same* splitter object to both `cross_val_score` calls, so fold 3 holds exactly the same rows for both models, and compute the per-fold gain

$$d_i \;=\; \text{score}_{A,\,i} - \text{score}_{B,\,i}, \qquad A = \text{challenger}, \;\; B = \text{current champion}.$$

Subtracting per fold is what makes the comparison sharp. Section 2's fold 5 was a hard slice of tracts — it would be hard for *any* model — and subtracting removes that shared difficulty, leaving only the part of each score that is genuinely about the models. Higher must mean better: use `'r2'`, `'roc_auc'`, or `'neg_mean_absolute_error'`.

**Step 2 — put an interval around the mean gain, corrected for fold overlap.** `compare_paired` computes

$$\bar{d} \;\pm\; t_{0.975,\,k-1}\cdot \text{SD}(d)\cdot\sqrt{\tfrac{1}{k} + \tfrac{1}{k-1}}.$$

The $\tfrac{1}{k}$ term is the familiar one. The extra $\tfrac{1}{k-1}$ term (the Nadeau–Bengio correction) accounts for the training rows the folds share — the dependence Section 1 warned about — and widens the interval by a factor of 1.5 at $k = 5$. At $k = 5$ the half-width works out to about $1.86 \times \text{SD}(d)$. Call the result an approximate 95% interval, corrected for fold overlap: a much better-calibrated instrument than an uncorrected one, but still an approximation.

**Step 3 — read the interval $[L, U]$ against the margin $\delta$ you declared before scoring.** Exactly one of four outcomes applies:

| `compare_paired` prints | Condition | What you do |
|---|---|---|
| **A is better by a worthwhile amount** | $L > \delta$ | Promote the challenger — the only outcome that changes the champion |
| **B is better by a worthwhile amount** | $U < -\delta$ | Keep the champion; the challenger is worse by more than the margin |
| **Practically equivalent: choose the simpler model** | $-\delta < L$ and $U < \delta$ | Keep the champion: any difference is smaller than the margin |
| **Unresolved: apply the default you declared** | anything else | Keep the champion, as the policy you declared in advance, and say so |

**The burden of proof sits with the challenger.** The model already running carries no switching cost and has already been explained to the stakeholder, so in this course "the simpler model" and "the default you declared" both mean: keep B. Only the third outcome earns the words *practically equivalent*, because only there does the interval rule out a gain as large as the margin. In the fourth, the honest sentence is *"the evidence does not show a gain large enough to justify the challenger, so under our policy we keep the champion"* — never "the models are equivalent".

**Fold wins are a description, never the verdict.** "The challenger won 5 of 5 folds" is worth reporting, but with no real difference and independent folds a named challenger wins all five by chance 1 time in 32, and the correlation CV folds have makes it more likely. A challenger can also win every fold by a hair and still be unresolved — the cell below shows one.

**Step 4 — run the band.** The champion comes from the **base** margin. Then repeat the comparison at the small and large margins: if the champion is the same at all three, report it as *robust across the three declared margins (for these folds)*; if it changes, report it as *margin-sensitive* and name the assumption that would flip it.

**Step 5 — write the champion statement.** Every verdict in this course ends as four points in the stakeholder's terms:

> **Champion.** We recommend **[model]** for **[stakeholder]'s [decision]**.
> **Evidence.** Against **[current champion]**, on the same five folds, it **[outcome]**: mean gain **[M] [metric]** (about **[business translation]**), interval corrected for fold overlap **[L, U]**, against the **δ = [value]** we set before comparing.
> **Checks.** [the small and large margins; anything else the stakeholder asked to see — fit time, interpretability, errors in dollars].
> **What would change our mind.** **[Runner-up]** is the candidate for a future evaluation (not an automatic fallback); we revisit if **[trigger]**.

The cell below runs `compare_paired` on four textbook sets of per-fold gains (each compared with a champion whose gain is zero, at $\delta = 0.01$) so you can see all four outcomes side by side. Then it asks the MedScreen question from Section 3: what is the best a **perfect** challenger — one scoring 1.0 on every fold — could show against the logistic model's five folds?

In [ ]:
# The four outcomes on four textbook sets of per-fold gains (challenger minus champion), δ = 0.01
examples = {
    'Clear gain':          [0.030, 0.025, 0.035, 0.028, 0.032],
    'Tiny, mixed signs':   [0.002, -0.001, 0.001, 0.003, 0.000],
    'Small, all positive': [0.001, 0.001, 0.001, 0.001, 0.010],
    'Clear loss':          [-0.030, -0.025, -0.035, -0.028, -0.032],
}
rows = []
for name, gains in examples.items():
    d, gain, (lo, hi), verdict = compare_paired(np.array(gains), np.zeros(5), 0.01)
    rows.append({'case': name, 'folds won by A': int((d > 0).sum()), 'mean gain': gain,
                 'low': lo, 'high': hi, 'outcome': verdict})
four = pd.DataFrame(rows)
print(four.to_string(index=False, formatters={c: '{:+.4f}'.format for c in ['mean gain', 'low', 'high']}))

fig, ax = plt.subplots(figsize=(11, 4))
ax.axvspan(-0.01, 0.01, color='gray', alpha=0.15, label='±δ (δ = 0.01, declared before scoring)')
ax.axvline(0, color='black', linewidth=0.8)
for i, r in four.iterrows():
    ax.errorbar(r['mean gain'], i, xerr=[[r['mean gain'] - r['low']], [r['high'] - r['mean gain']]],
                fmt='o', color='steelblue', ecolor='steelblue', capsize=6, markersize=8)
    ax.text(0.045, i, r['outcome'].split(':')[0], va='center', fontsize=9)
ax.set_yticks(range(len(four)))
ax.set_yticklabels(four['case'])
ax.invert_yaxis()
ax.set_xlim(-0.045, 0.09)
ax.set_xlabel('Mean gain of the challenger (A − B), with the interval corrected for fold overlap')
ax.set_title('Four outcomes, read against ±δ', fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# The MedScreen ceiling: the best a PERFECT challenger could show against the Section 3 folds
perfect = np.ones(k)
d_p, gain_p, (lo_p, hi_p), verdict_p = compare_paired(perfect, cv_scores_clf, DELTA_AUC)
print("\n=== MedScreen: a perfect challenger (AUC = 1.0 on every fold) vs LogReg (C=1) ===")
print(f"Gain per fold: {np.round(d_p, 4)}")
print(f"Mean gain {gain_p:+.4f}, interval corrected for fold overlap [{lo_p:+.4f}, {hi_p:+.4f}] → {verdict_p}")

**Reading the output:**

The four rows of the table and the four lines of the figure are the whole decision vocabulary of the course, so it is worth reading them slowly.

**Clear gain** — five gains between 0.025 and 0.035, mean **+0.0300**, interval **[+0.0229, +0.0371]**. The whole interval sits above $+\delta$: the challenger is better by a worthwhile amount, and it is promoted. **Clear loss** is the mirror image, **[−0.0371, −0.0229]**: the champion is better by a worthwhile amount and stays.

**Tiny, mixed signs** — mean **+0.0010**, interval **[−0.0019, +0.0039]**, entirely inside the gray $\pm\delta$ band: **practically equivalent**. This is a *positive* finding — the evidence rules out a gain as large as the margin in either direction — and the champion stays.

**Small, all positive** is the row that matters most. The challenger **wins all five folds**, yet the mean is only **+0.0028** and the interval **[−0.0047, +0.0103]** pokes past $+\delta$ on one side and below zero on the other: **unresolved**. Four tiny wins and one larger one are not enough to show a gain the stakeholder said would matter, so under the declared policy the champion stays. Counting fold wins would have crowned the challenger; the interval does not.

**The MedScreen ceiling.** Against the logistic model's five folds, a perfect challenger gains **0.0000, 0.0084, 0.0093, 0.0000, 0.0147** per fold — exactly the headroom printed in Section 3 — for a mean of **+0.0065** and an interval of **[−0.0054, +0.0183]**: **unresolved**. Even a model that ranks every patient perfectly could not show the board a worthwhile gain on these folds, because two folds leave nothing to gain. That is the honest summary for the chief medical officer: on this evidence the comparison cannot crown any challenger, so the explainable model stays by policy.

> **A question that often comes up here:** *"If the margin is just an assumption, why not pick the one that makes my favorite model win?"* Because a margin chosen after you have seen the scores is not a business assumption any more; it is a way of dressing up the answer you already wanted. Writing down three margins before you score, and letting only the middle one decide, protects you from that temptation and still shows the stakeholder how much the decision depends on how demanding they are. When the answer does depend on the margin, say which assumption would flip it — and keep the current champion unless the stakeholder told you in advance that the other assumption is the one that applies.

**Key takeaway:** one splitter, per-fold gains, an interval corrected for fold overlap, a margin declared in advance, four outcomes, and only one of them promotes the challenger. The two exercises now put the tool to work on HomeValue and MedScreen.

---

## 📝 PAUSE-AND-DO Exercise 1 (10 minutes)

**Task:** Does Ridge earn the job over plain OLS on California Housing?

This is the question the CFO has actually been asking since nb05. HomeValue runs **plain OLS** today — it has no hyperparameter to defend — so OLS is the **current champion (B)** and Ridge is the **challenger (A)**. You reached for Ridge because regularization seemed like a good idea; can you *defend* that choice to a skeptical finance board using only the evidence CV gives you?

Run the same 5-fold CV recipe from Section 2 with a plain `LinearRegression` pipeline (no L2 penalty), keeping everything else identical — same `X_reg_train`, the same `cv` splitter object from Section 2, same `scoring='r2'` — and report OLS's fold scores, mean, and SD. Because both runs pass the *identical* `KFold` object, fold 3 holds exactly the same tracts for both models, which licenses the step that settles the question: `compare_paired(cv_scores, ols_scores, DELTA_R2)`, Ridge first.

**Decision rule (this is the one you will use for the rest of the course):** the margins are already on paper from Section 2, and the **base margin, 0.01 $R^2$**, decides. Ridge replaces OLS only if `compare_paired` prints *"A is better by a worthwhile amount"*. In the other three outcomes OLS stays — as the simpler model if the two are practically equivalent, as the declared default if the comparison is unresolved, and obviously if OLS is better by a worthwhile amount. Then run the band: repeat the comparison at the small (0.005) and large (0.02) margins and report whether the champion is the same at all three (robust) or changes (margin-sensitive). Finally, draw the comparison: the two models' fold scores joined fold by fold, and the mean gain with its interval against the $\pm\delta$ band.

> **A question that often comes up on the first paired comparison:** *"If the challenger wins most of the folds, doesn't that settle it?"* No — fold wins are a description, never the verdict. With no real difference at all, a named challenger wins all five folds by chance about 1 time in 32, and more often than that when the folds share training rows; and a challenger can win every fold by a hair, as the "small, all positive" row in Section 4 did, and still fall short of a gain the CFO said would matter. The interval around the mean gain, read against the margin you wrote down before scoring, is what decides — report the fold wins beside it as color, not as evidence.

---

> 💡 **Gemini Prompt:** "Build a plain linear regression pipeline (StandardScaler + LinearRegression). Using the same `X_reg_train`, `y_reg_train` and the same `cv` KFold object from Section 2, run `cross_val_score` with `scoring='r2'` and store the five fold scores as `ols_scores`. Print them with their mean and sample SD (`ddof=1`). Then call `compare_paired(cv_scores, ols_scores, DELTA_R2)` — Ridge is the challenger (A), OLS the current champion (B) — and print the five per-fold gains with 6 decimals, their mean, the interval, and the outcome. Repeat the call for each margin in `DELTAS_R2` and print one line per margin with the outcome and the resulting champion (Ridge only if the outcome starts with 'A is better'). Finally, draw a 1x2 figure: left, the two models' fold scores with a line joining each fold's pair; right, the mean gain as a dot with a horizontal error bar for the interval, a shaded band from `-DELTA_R2` to `+DELTA_R2`, and a vertical line at zero."
>
> **After running, verify:**
> - OLS's five fold scores, mean, and SD are printed with 4-decimal precision
> - The five per-fold gains (Ridge − OLS) and their mean are printed with 6 decimals, plus the interval and one of the four outcomes at the base margin
> - One line per margin (small, base, large) shows its outcome and the champion it implies
> - Left panel: two dots per fold joined by a line; right panel: one dot with a horizontal error bar inside or across the shaded ±δ band
> - No error bars are drawn around either model's own score
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build a plain OLS pipeline: StandardScaler + LinearRegression
#   2. Run 5-fold CV on X_reg_train, y_reg_train with scoring='r2'
#      (pass the SAME cv object from Section 2, so the folds line up row for row)
#   3. Print OLS's fold scores, mean, and sample SD (ddof=1)
#   4. compare_paired(cv_scores, ols_scores, DELTA_R2): Ridge = A (challenger), OLS = B (champion)
#      Print the per-fold gains, their mean, the interval, and the outcome
#   5. Run the band: repeat for each margin in DELTAS_R2; Ridge is champion only if the
#      outcome starts with 'A is better'. Same champion at all three = robust
#   6. Plot: paired fold scores (left) and the mean gain with its interval vs ±δ (right)
#
# Variables available from Sections 2 and 4:
#   - X_reg_train, y_reg_train
#   - cv (the 5-fold KFold splitter with random_state=RANDOM_SEED)
#   - cv_scores (the five Ridge per-fold R² values)
#   - compare_paired, DELTAS_R2, DELTA_R2 (from the setup cell)

### YOUR ANALYSIS:

**Question 1:** Which of the four outcomes does `compare_paired` print at the base margin, and does the champion change at the small or the large margin?
[Your answer — read the interval against ±δ, not the number of folds each model won]

**Question 2:** Write the four-point champion statement for the CFO (Champion, Evidence, Checks, What would change our mind).
[Your answer]

**Question 3:** The interval uses $\sqrt{1/k + 1/(k-1)}$ instead of the textbook $\sqrt{1/k}$. What would happen to the interval — and to the outcomes you could reach — if you dropped the $1/(k-1)$ term? Recall from Section 1 why the fold scores are not independent.
[Your answer]

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Should MedScreen switch from the default `LogisticRegression(C=1.0)` to a strongly regularized `C=0.01`?

Exercise 1 compared *algorithms* (Ridge vs. plain OLS). This exercise asks a subtly different question: with the same algorithm, does changing the *hyperparameter* earn a switch? The review board runs the default `C=1.0` today (the model from nb06–nb07), so `C=1.0` is the **current champion (B)** and `C=0.01` is the **challenger (A)**. The tool stays identical — that is the whole point.

Rerun the CV recipe from Section 3 on `X_clf_train` / `y_clf_train` with `LogisticRegression(C=0.01)`, keeping the `StandardScaler`, the same `cv_strat` splitter object from Section 3 (so fold $i$ holds the same patients for both settings), and `scoring='roc_auc'`. Report the challenger's fold scores, mean, and SD, then call `compare_paired(strong_scores, cv_scores_clf, DELTA_AUC)` — challenger first.

**Decision rule:** the same as Exercise 1, with the board's margins from Section 3 — the **base margin, 0.01 ROC-AUC**, decides, and `C=0.01` replaces `C=1.0` only on *"A is better by a worthwhile amount"*. Run the band at 0.005 and 0.02, draw the same two panels as in Exercise 1, and keep Section 3's headroom in mind when you read the interval.

> **A question that often comes up here:** *"Isn't this what `GridSearchCV` does automatically?"* Almost — and that is nb09's entire topic. `GridSearchCV` scores a whole grid of candidate values on the identical folds and ranks them by mean CV score, but the top-ranked row is a **nominee**, not a new champion: in nb09 it faces the default setting with exactly this `compare_paired` call and replaces it only on a worthwhile gain. There is a second reason to be careful with the top row — it was picked *because* it scored highest on these folds, so its score is a little optimistic. The reason you are doing the comparison by hand today is so that when nb09 automates the search, you already know what the machine does and does not decide for you.

---

> 💡 **Gemini Prompt:** "Build a logistic regression pipeline with StandardScaler + LogisticRegression(C=0.01, random_state=RANDOM_SEED, max_iter=5000). Using the same `X_clf_train`, `y_clf_train`, and the same `cv_strat` StratifiedKFold object from Section 3, run `cross_val_score` with `scoring='roc_auc'` and store the five fold scores as `strong_scores`. Print them with their mean and sample SD (`ddof=1`). Then call `compare_paired(strong_scores, cv_scores_clf, DELTA_AUC)` — C=0.01 is the challenger (A), C=1.0 the current champion (B) — and print the per-fold gains with 4 decimals, their mean, the interval, and the outcome. Repeat the call for each margin in `DELTAS_AUC` and print one line per margin with the outcome and the resulting champion (C=0.01 only if the outcome starts with 'A is better'). Finally, draw a 1x2 figure: left, the two settings' fold scores with a line joining each fold's pair; right, the mean gain as a dot with a horizontal error bar for the interval, a shaded band from `-DELTA_AUC` to `+DELTA_AUC`, and a vertical line at zero."
>
> **After running, verify:**
> - The challenger's five fold AUCs, mean, and SD are printed with 4-decimal precision
> - The five per-fold gains (C=0.01 − C=1.0) and their mean are printed, plus the interval and one of the four outcomes at the base margin
> - One line per margin (small, base, large) shows its outcome and the champion it implies
> - Left panel: two dots per fold joined by a line; right panel: one dot with a horizontal error bar against the shaded ±δ band
> - No error bars are drawn around either setting's own score
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build the challenger: StandardScaler + LogisticRegression(C=0.01, max_iter=5000)
#   2. Run 5-fold STRATIFIED CV on X_clf_train, y_clf_train with scoring='roc_auc'
#      (pass the SAME cv_strat object from Section 3)
#   3. Print the challenger's fold scores, mean, and sample SD (ddof=1)
#   4. compare_paired(strong_scores, cv_scores_clf, DELTA_AUC): C=0.01 = A (challenger), C=1.0 = B (champion)
#      Print the per-fold gains, their mean, the interval, and the outcome
#   5. Run the band over DELTAS_AUC; C=0.01 is champion only if the outcome starts with 'A is better'
#   6. Plot: paired fold scores (left) and the mean gain with its interval vs ±δ (right)
#
# Variables available from Sections 3 and 4:
#   - X_clf_train, y_clf_train
#   - cv_strat (the 5-fold StratifiedKFold splitter with random_state=RANDOM_SEED)
#   - cv_scores_clf (the five C=1.0 per-fold AUCs)
#   - compare_paired, DELTAS_AUC, DELTA_AUC (from the setup cell)

### YOUR ANALYSIS:

**Question 1:** Which of the four outcomes does `compare_paired` print at the base margin, and what does the band (0.005 / 0.01 / 0.02) say about how robust the choice of champion is?
[Your answer — read the interval against ±δ, not the point estimates]

**Question 2:** Write the four-point champion statement for the review board. If the outcome is unresolved, use the sentence the course prescribes for it.
[Your answer]

**Question 3:** nb09 will search a whole grid of `C` values. Why does the grid's best row become a *nominee* that faces `C=1.0` with `compare_paired`, rather than the new champion?
[Your answer]

---

## 5. The Pattern From nb09 On — One 80/20 Split and Five Folds

The 60/20/20 split did its job today: it showed you that one validation score can run high — HomeValue's single split sat 0.03 above the CV mean, above every one of the five folds. Once you have cross-validation, a separate validation set has nothing left to do that five folds on more rows do not do better. So from nb09 on, every notebook works the same way:

- **One split per dataset, 80/20**, with the same first call you have used since nb01 (`test_size=0.20`, `random_state=474`, stratified for classification). Because the first call is unchanged, the locked test rows are *exactly the same rows* as before; the old training and validation rows simply merge into one larger `X_train`.
- **All model work on `X_train` with five folds** — comparisons with `compare_paired`, grid searches, threshold choices, probability checks — using the two splitters defined below, `cv_reg` and `cv_clf`. No validation set, no second split, no rows set aside inside `X_train`.
- **`X_test` stays locked** until nb14, where it opens once per business case, for the frozen champion only.

The cell below sets up that pattern and checks the two claims in the first bullet.

In [ ]:
# The one split every notebook from nb09 on makes: 80/20, seed 474, stratified for classification
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)

# Five folds everywhere: the two splitters every later notebook passes to cross_val_score and compare_paired
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f"HomeValue: X_train {len(X_train_reg):,} rows | X_test {len(X_test_reg):,} rows (locked)")
print(f"MedScreen: X_train {len(X_train_clf):,} rows | X_test {len(X_test_clf):,} rows (locked)")
same_test = X_test_reg.index.equals(X_reg_test.index) and X_test_clf.index.equals(X_clf_test.index)
merged = (set(X_train_reg.index) == set(X_reg_train.index) | set(X_reg_val.index)
          and set(X_train_clf.index) == set(X_clf_train.index) | set(X_clf_val.index))
print(f"Same locked test rows as the 60/20/20 split above:       {same_test}")
print(f"X_train = the old training rows + the old validation rows: {merged}")

**Reading the output:**

HomeValue now has **16,512** training rows and the same **4,128** locked test tracts; MedScreen has **455** training patients and the same **114** locked test patients. Both checks print **True**: dropping the second split moved no test row, and every row that used to sit in a validation set is now in `X_train`, where five folds put it to work for fitting *and* for evaluation. Every comparison from nb09 on runs on these `X_train` sets with `cv_reg` or `cv_clf`, so its fold scores — and the `compare_paired` intervals built from them — are comparable across notebooks.

> **A question that often comes up here:** *"Without a validation set, how do I check a model before the test set?"* With the same five folds you already used today. Held-out predictions for a classification report or a threshold come from `cross_val_predict` on `X_train`, where every row is predicted by a model that never saw it; comparisons come from `compare_paired` on fold scores; tuning runs a grid search whose folds live inside `X_train`. The test set is not a checking tool at all — it is opened once, in nb14, for the champion you have already frozen, to confirm it meets the deployment bar the stakeholder declared in advance.

---

## 6. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **One split is one draw from a distribution.** HomeValue's single validation split scored 0.6169, above all five fold scores; $k$-fold CV produces $k$ numbers, and their mean and spread are a far more honest estimate of how the model performs.

2. **The CV equation is the same across problem types.** Regression averages per-fold MSE; classification averages per-fold misclassification. Accuracy and $R^2$ are direct rewrites of those losses; ROC-AUC (a ranking measure) and F1 (a one-threshold balance of precision and recall) are not, but they are computed fold by fold and averaged in exactly the same way. The two classification ingredients: **stratification** (preserve class prevalence in every fold) and a **score** that matches the business question from nb07.

3. **Report the fold scores, their mean, and their SD — and show them.** No 95% interval around one model's score: the fold scores share training rows, so that interval runs too narrow, and the stakeholder's question is never about one model in isolation.

4. **A single split against the CV mean is a description.** INSIDE / ABOVE / BELOW says whether one split landed within the declared margin of the CV mean. It never proves that a model generalizes; whether a model ships is decided in nb14 against a deployment bar declared in advance.

5. **Compare models with `compare_paired`.** Challenger (A) against current champion (B), on the identical folds, per-fold gains, an interval corrected for fold overlap, read against the margin. **Four outcomes** — A better by a worthwhile amount, B better by a worthwhile amount, practically equivalent, unresolved — and only the first promotes the challenger. Fold wins are color, not evidence.

6. **Declare the margins before scoring, as a band.** The base margin decides; small and large say whether the champion is robust or margin-sensitive — check which of the two your Exercise 1 and Exercise 2 champions turned out to be.

7. **Write the champion statement.** Champion, evidence (outcome, mean gain, interval, margin), checks, and what would change your mind. For an unresolved result: *"the evidence does not show a gain large enough to justify the challenger, so under our policy we keep the champion."*

8. **From nb09 on: one 80/20 split, five folds, no validation set.** The locked test rows are the same rows as before; everything else happens inside `X_train`.

### Critical Rules:

> **"Never trust a single split — always cross-validate before a model-choice decision."**

> **"Quote the CV mean with its fold scores and SD — never the mean alone, and never a per-model interval."**

> **"Stratify folds for classification so class balance survives every fold."**

> **"Same folds, per-fold gains, the corrected interval against a margin declared in advance: only a worthwhile gain promotes the challenger."**

### Next Steps:

- **nb09** turns today's CV ritual into a whole search. `GridSearchCV` runs the Section 2–3 machinery once per hyperparameter candidate and ranks the results; the top row is a **nominee** that faces the default with `compare_paired`, exactly as `C=0.01` faced `C=1.0` today. If today felt repetitive, that is a good sign — the repetition is the point, because nb09 is today's comparison, automated over a grid.
- **Your final project** should use this framework end-to-end: declare the stakeholder's margins before scoring, report each candidate's fold scores with their mean and SD, compare every challenger with the current champion using `compare_paired` on the same five folds, and close with a champion statement. If you can hand your reviewer the margins, the per-fold gains, the interval, and the outcome, you are already most of the way to a professional evaluation section.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Model Assessment and Selection (k-fold CV, resampling concepts)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Resampling theory and selection bias
- scikit-learn User Guide: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- Kohavi, R. (1995). "A study of cross-validation and bootstrap for accuracy estimation and model selection." *IJCAI*.
- Nadeau, C., & Bengio, Y. (2003). Inference for the generalization error. *Machine Learning*, 52 — the correction for fold overlap used in `compare_paired`
- Bouckaert, R. R., & Frank, E. (2004). Evaluating the replicability of significance tests for comparing learning algorithms. *PAKDD 2004*, 3–12
- Bengio, Y., & Grandvalet, Y. (2004). No unbiased estimator of the variance of k-fold cross-validation. *Journal of Machine Learning Research*, 5
- Bates, S., Hastie, T., & Tibshirani, R. (2024). Cross-validation: What does it estimate and how well does it do it? *Journal of the American Statistical Association*, 119(546), 1434–1445
- Lakens, D. (2017). Equivalence tests: A practical primer for t tests, correlations, and meta-analyses. *Social Psychological and Personality Science*, 8(4), 355–362 — why "practically equivalent" needs the whole interval inside the margin

---

<center>

Thank you!

</center>